Edge ai-powered Real time violation detection using YOLOV8

In [ ]:
!pip install ultralytics opencv-python-headless tqdm

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import cv2
video_path = '/content/CCTV Footage.mp4'
cap = cv2.VideoCapture(video_path)


In [ ]:
from ultralytics import YOLO
from IPython.display import display, Image
from PIL import Image as PILImage
import numpy as np

model = YOLO("yolov8n.pt")  # You can switch to yolov8s/m/l for more accuracy

frame_count = 0
max_frames = 100  # For demo purposes (limit to save time)

while cap.isOpened() and frame_count < max_frames:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)

    for box in results[0].boxes.data:
        x1, y1, x2, y2, conf, cls = box
        label = model.names[int(cls)]
        if label in ['car', 'bus', 'truck']:
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0,255,0), 2)
            cv2.putText(frame, label, (int(x1), int(y1)-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    # Display frame (convert from BGR to RGB)
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    display(PILImage.fromarray(img))

    frame_count += 1


In [ ]:
import numpy as np
from collections import defaultdict
from datetime import timedelta
import cv2 # Make sure cv2 is imported

# Initialize variables
previous_positions = {}
speed_limit = 30  # Pixel/frame (you can later convert to km/h)
STOP_LINE_Y = 250  # Change as per video (you'll see a horizontal line)
signal_status = "RED"  # Simulate a red light

# Function to calculate speed
def calculate_speed(vehicle_id, center, frame_num):
    if vehicle_id not in previous_positions:
        previous_positions[vehicle_id] = (center, frame_num)
        return 0

    prev_center, prev_frame = previous_positions[vehicle_id]
    distance = np.linalg.norm(np.array(center) - np.array(prev_center))
    frames_elapsed = frame_num - prev_frame
    speed = distance / frames_elapsed if frames_elapsed > 0 else 0
    previous_positions[vehicle_id] = (center, frame_num)
    return speed

# Redraw the detection loop with speed + signal jumping
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)  # Reset video
frame_num = 0
frame_rate = cap.get(cv2.CAP_PROP_FPS) # Get frame rate from the video

while cap.isOpened() and frame_num < max_frames:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)
    frame_time = timedelta(seconds=frame_num / frame_rate)

    for i, box in enumerate(results[0].boxes.data):
        x1, y1, x2, y2, conf, cls = box
        label = model.names[int(cls)]
        if label in ['car', 'bus', 'truck']:
            center = ((x1 + x2) / 2, (y1 + y2) / 2)
            vehicle_id = f"{label}_{i}"  # Fake ID for demo

            speed = calculate_speed(vehicle_id, center, frame_num)
            cv2.putText(frame, f"Speed: {speed:.1f}", (int(x1), int(y2)+20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,0), 2)

            if speed > speed_limit:
                print(f"[{frame_time}] 🚨 Speeding detected for {label}, Speed: {speed:.1f}")

            if signal_status == "RED" and y2 < STOP_LINE_Y:
                print(f"[{frame_time}] 🚨 Signal jump detected for {label}")

            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0,255,0), 2)
            cv2.putText(frame, label, (int(x1), int(y1)-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    # Draw stop line
    cv2.line(frame, (0, STOP_LINE_Y), (frame.shape[1], STOP_LINE_Y), (0, 0, 255), 2)

    # Display
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    display(PILImage.fromarray(rgb))

    frame_num += 1